# Matrix factorization

Run all cells in a kernel with [GALAHAD GLRT installed](CUBIC_SUBPROBLEM.md).
The cubic comparison now uses GLRT instead of the former dense solver.
Data, initialization and outer acceptance are unchanged; the inexact solve
can change trajectories. Rerun to regenerate the figures.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

%reload_ext autoreload
%autoreload 2
%matplotlib inline

In [ ]:
from methods_mat_fac import mf_terms_and_metrics
from oracles_mat_fac import PositiveMatrixFactorization
from methods import gradient_method, mf_gladssn, mf_leapssn, cubic_newton_nonconvex_new
from utils import run_methods, plot_figures

# Matrix Factorisation

### Parameters

In [ ]:
np.random.seed(0)

# Ground truth
m, n, r = 200, 100, 12
U_true = np.random.rand(m, r)
V_true = np.random.rand(n, r).T  # (r, n); preserve the previous random draw
M_clean = U_true @ V_true

# Noisy observed matrix
noise = 0.0
M_obs = M_clean + noise * np.random.randn(m, n)

# Revised equation (27): alpha/2 ridge, beta/2 nonnegativity penalty
alpha = 1e-2
beta = 200.0  # beta/2 = 100, preserving the previous penalty weight

# Initialise
# Preserve previous starting factors, then repack V in the paper orientation.
initial_values = 0.5 * np.random.randn(m * r + n * r)
U_0 = initial_values[:m*r].reshape(m, r)
V_0 = initial_values[m*r:].reshape(n, r).T


# Metrics function (same for both)
def make_metric_fn(oracle):
    return lambda z: mf_terms_and_metrics(
        oracle, z, M_clean=M_clean, U_true=U_true, V_true=V_true
    )


oracle = PositiveMatrixFactorization(M_obs, r=r, alpha=alpha, beta=beta)
x_0 = oracle.pack(U_0, V_0)


In [ ]:
number_of_iterations = 200 #100#6500  # 6500  # 12000 #8000 #350

# Run methods

In [ ]:
methods_to_run = {
    "Gradient Method": (gradient_method, (oracle, x_0), dict(max_iter=10000)), # 3000000
    "Cubic Newton": (
    cubic_newton_nonconvex_new,
    (oracle, x_0),
    dict(
        n_iters=number_of_iterations,
        H_0=1,
        warnings=True,
        subproblem_tol=1e-8,
        glrt_max_iter=6000,
        max_backtracks=119,
    ),
),
    "LeAP-SSN": (
        mf_leapssn,
        (oracle, x_0),
        dict(
            n_iters=number_of_iterations,
            H_0=1,
            warnings=True,
            lss_tol=1e-9,
            lss_method="minres",
            use_gauss_newton=False,
        ),
    ),
        "GLAd-SSN, p=0.2": (
        mf_gladssn,
        (oracle, x_0),
        dict(
            n_iters=number_of_iterations,
            H_0=1,
            warnings=True,
            p=0.2,
            lss_tol=1e-9,
            lss_method="minres",
            use_gauss_newton=False,
        ),
    ),
        "GLAd-SSN, p=0.4": (
        mf_gladssn,
        (oracle, x_0),
        dict(
            n_iters=number_of_iterations,
            H_0=1,
            warnings=True,
            p=0.4,
            lss_tol=1e-9,
            lss_method="minres",
            use_gauss_newton=False,
        ),
    ),
    "GLAd-SSN, p=0.5": (
        mf_gladssn,
        (oracle, x_0),
        dict(
            n_iters=number_of_iterations,
            H_0=1,
            warnings=True,
            p=0.5,
            lss_tol=1e-9,
            lss_method="minres",
            use_gauss_newton=False,
        ),
    ),
    "GLAd-SSN, p=0.6": (
        mf_gladssn,
        (oracle, x_0),
        dict(
            n_iters=number_of_iterations,
            H_0=1,
            warnings=True,
            p=0.6,
            lss_tol=1e-9,
            lss_method="minres",
            use_gauss_newton=False,
        ),
    ),
        "GLAd-SSN, p=0.8": (
        mf_gladssn,
        (oracle, x_0),
        dict(
            n_iters=number_of_iterations,
            H_0=1,
            warnings=True,
            p=0.8,
            lss_tol=1e-9,
            lss_method="minres",
            use_gauss_newton=False,
        ),
    ),
    
}
histories = run_methods(methods_to_run, oracle, x_0)

# GM ONLY!

In [ ]:
# GM_methods_to_run = {
#     "Gradient Method": (gradient_method, (oracle, x_0), dict(max_iter=3000000)),
    
# }
# new_GM_histories = run_methods(GM_methods_to_run, oracle, x_0)

In [ ]:
 

# histories["Gradient Method"] = new_GM_histories["Gradient Method"]

# Glad only

In [ ]:
# methods_to_run = {
#         "GLAd-SSN, p=0.9": (
#         mf_gladssn,
#         (oracle, x_0),
#         dict(
#             n_iters=number_of_iterations,
#             H_0=1,
#             warnings=True,
#             p=0.8,
#             lss_tol=1e-9,
#             lss_method="minres",
#             use_gauss_newton=False,
#         ),
#     ),
    
# }
# NEW_histories = run_methods(methods_to_run, oracle, x_0)

In [ ]:
# histories["GLAd-SSN, p=0.9"] = NEW_histories["GLAd-SSN, p=0.9"]


In [ ]:
# histories.pop("GLAd-SSN, p=0.9")  # Remove CN for better visualisation
# print(histories.keys())

# Code for plotting

In [ ]:


line_widths = [2.5] * len(methods_to_run)
line_styles = ["-"] * len(methods_to_run)

gradient_method_color = "#E8601C"
cubic_newton_color = "gold"
leapssn_color = "black"
gladssn_color = "#9B8AC4"
gladssn_1_color = "#88A5DD"
gladssn_2_color = "#8DCBE4"
gladssn_3_color = "#C2E3D2"
gladssn_4_color = "#DDECBF"
# gladssn_5_color = "#F2E1D0"
colors = [gradient_method_color, cubic_newton_color,leapssn_color, gladssn_color, gladssn_1_color, gladssn_2_color, gladssn_3_color, gladssn_4_color]
marker_cycle = [
"None",   # None Gradient Method
"*",   # star Cubic Newton
"s",   # square LeapSSN
"^",   # triangle up GladSSN
"D",   # diamond GladSSN second
"x",   # x GladSSN third
"v",   # triangle down GladSSN fourth
"h",   # hexagon GladSSN fifth
] 


list_of_mins = [np.min(history["func"]) for history in histories.values()]
mn_func = min(list_of_mins)
plot_figures(
    list(histories.values()),
    list(histories.keys()),
    colors=colors,
    linewidths=line_widths,
    linestyles=line_styles,
    n=100,
    m=None,
    mu_str=("0.5"),
    f_star=mn_func,  # mn_func,
    threshold=1e-10,
    suptitle=("Matrix Factorization"),
    which_measure_2="func",
    max_iter=1000,
)
plot_figures(
    list(histories.values()),
    list(histories.keys()),
    colors=colors,
    linewidths=line_widths,
    linestyles=line_styles,
    n=100,
    m=None,
    mu_str=("0.5"),
    f_star=0,
    threshold=1e-10,
    suptitle=("Matrix Factorization"),
    max_iter=1000,
    which_measure_1="grad_norm",
    which_measure_2="grad_norm",
    # which_measure_2="grad_norm_at_lssolve",
    ylabel="Gradient norm",
)

plt.show()

# Plots for the paper

In [ ]:
from utils import plot_for_paper_2, plot_for_paper_penalty
markevery_numbers = [50000, 1000, 1000, 1000, 1000, 1000, 1000, 1000]

plot_for_paper_2(
    list(histories.values()),
    list(histories.keys()),
    colors=colors,
    linewidths=line_widths,
    linestyles=line_styles,
    n=100,
    m=None,
    mu_str=("0.5"),
    f_star=mn_func,
    threshold=1e-10,
    suptitle=("Plots"),
    max_iter=12000,
    which_measure="func",
    marker_cycle=marker_cycle,
    markevery_numbers=markevery_numbers,
)
plot_for_paper_2(
    list(histories.values()),
    list(histories.keys()),
    colors=colors,
    linewidths=line_widths,
    linestyles=line_styles,
    n=100,
    m=None,
    mu_str=("0.5"),
    f_star=mn_func,
    threshold=1e-10,
    suptitle=("Plots"),
    max_iter=12000,
    which_measure="grad_norm",
    marker_cycle=marker_cycle,
    markevery_numbers=markevery_numbers,
)
plot_for_paper_penalty(
    list(histories.values()),
    list(histories.keys()),
    colors=colors,
    linewidths=line_widths,
    linestyles=line_styles,
    marker_cycle=marker_cycle,
    max_time=12000,
)

In [ ]:
# Save only the series used by the MF plots.
import pickle

plot_keys = ("func", "time", "grad_norm", "penalty")
plot_histories = {
    label: {key: np.asarray(history[key]) for key in plot_keys if key in history}
    for label, history in histories.items()
}
with open("mf_results.pkl", "wb") as result_file:
    pickle.dump({"histories": plot_histories, "labels": list(histories)}, result_file)
print("Saved plotting data to mf_results.pkl")